# 📊 Dataset Preparation — Flood Rescue AI System

> **Project**: Edge AI–Based System for Multimodal Analysis and Clustering of Flood Rescue Events
>
> **Purpose**: Prepare image + text datasets for Progress Report #1
>
> **Environment**: Google Colab (Free) / Kaggle Notebook

## 1. Environment Setup

In [ ]:
!pip install -q torch torchvision Pillow pandas matplotlib seaborn scikit-learn
!pip install -q icrawler gdown kaggle tqdm requests beautifulsoup4

In [ ]:
import os, sys, json, random, shutil, hashlib, time, requests, warnings, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
SEED = 42
random.seed(SEED); np.random.seed(SEED)

BASE_DIR = Path('dataset')
IMG_DIR = BASE_DIR / 'image_data'
TXT_DIR = BASE_DIR / 'text_data'
RAW_DIR = BASE_DIR / 'raw_downloads'
REPORT_DIR = BASE_DIR / 'reports'
CLASSES = ['no_flood', 'low_flood', 'high_flood']
TEXT_CLASSES = ['urgent_rescue', 'need_supplies', 'safe_update', 'irrelevant']

for d in [IMG_DIR, TXT_DIR, RAW_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        (IMG_DIR / split / cls).mkdir(parents=True, exist_ok=True)

SOURCE_URLS = {
    'FloodNet': 'https://github.com/BinaLab/FloodNet-Supervised_v1.0',
    'CrisisMMD': 'https://crisisnlp.qcri.org/crisismmd',
    'Kaggle': 'https://www.kaggle.com/search?q=flood+images',
    'Vietnam-collected': 'manual_collection',
    'synthetic_vietnamese': 'manual_authoring_for_progress_report',
}

def sha1_of_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha1()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print('✅ Environment ready!')
print(f'Base directory: {BASE_DIR.resolve()}')

## 2. FloodNet Dataset

Source: [FloodNet](https://github.com/BinaLab/FloodNet-Supervised_v1.0) — ~2,343 UAV images, 10 classes → remap to 3 classes.

In [ ]:
FLOODNET_REMAP = {
    'Flooded': 'high_flood', 'Building-Flooded': 'high_flood', 'Road-Flooded': 'high_flood',
    'Water': 'low_flood', 'Mud': 'low_flood',
    'Building-Non-Flooded': 'no_flood', 'Road-Non-Flooded': 'no_flood',
    'Grass': 'no_flood', 'Tree': 'no_flood', 'Vehicle': 'no_flood', 'Pool': 'no_flood',
}

floodnet_dir = RAW_DIR / 'floodnet'
floodnet_dir.mkdir(exist_ok=True)

print("📥 Downloading FloodNet dataset...")
print("⚠️  If auto-download fails, manually download from:")
print("    https://github.com/BinaLab/FloodNet-Supervised_v1.0")
print(f"    Extract to: {floodnet_dir.absolute()}")

try:
    import gdown
    url = "https://drive.google.com/drive/folders/1g1r416bRLNlpbKmEMg9heFVFBz-fFSaJ"
    gdown.download_folder(url, output=str(floodnet_dir), quiet=False, use_cookies=False)
    print("✅ FloodNet downloaded!")
except Exception as e:
    print(f"⚠️  Auto-download failed: {e}")
    print("Creating sample structure for demonstration...")
    for cls_name in FLOODNET_REMAP:
        d = floodnet_dir / cls_name; d.mkdir(exist_ok=True)
        for i in range(5):
            Image.new('RGB',(224,224),color=(random.randint(0,255),random.randint(0,255),random.randint(0,255))).save(d/f'{cls_name.lower()}_{i:04d}.jpg')

In [ ]:
def process_floodnet(src_dir):
    records = []
    src_path = Path(src_dir)
    for orig_class, mapped_class in FLOODNET_REMAP.items():
        class_dir = None
        for d in src_path.rglob('*'):
            if d.is_dir() and orig_class.lower() in d.name.lower():
                class_dir = d; break
        if class_dir is None:
            continue
        images = list(class_dir.glob('*.jpg'))+list(class_dir.glob('*.png'))+list(class_dir.glob('*.jpeg'))
        print(f"  {orig_class:25s} → {mapped_class:12s}: {len(images)} images")
        for p in images:
            records.append({'filename':p.name,'source':'FloodNet','original_label':orig_class,
                           'mapped_label':mapped_class,'is_vietnam':False,'original_path':str(p)})
    print(f"\n📊 Total FloodNet: {len(records)}")
    return records

floodnet_records = process_floodnet(floodnet_dir)
if floodnet_records:
    print("\nRemapped distribution:")
    print(pd.DataFrame(floodnet_records)['mapped_label'].value_counts())

## 3. CrisisMMD v2.0

Source: [CrisisNLP](https://crisisnlp.qcri.org/crisismmd) — filter flood events, remap damage_severity.

In [ ]:
CRISISMMD_REMAP = {
    'severe_damage':'high_flood','severe':'high_flood',
    'mild_damage':'low_flood','mild':'low_flood',
    'little_or_no_damage':'no_flood','little_or_none':'no_flood',
}
FLOOD_EVENTS = ['hurricane_harvey','hurricane_irma','hurricane_maria','sri_lanka_floods','srilanka_floods','flood']

crisismmd_dir = RAW_DIR / 'crisismmd'; crisismmd_dir.mkdir(exist_ok=True)
print("📥 CrisisMMD v2.0")
print("⚠️  Manual download: https://crisisnlp.qcri.org/crisismmd")

# Create sample
for ev in ['hurricane_harvey','sri_lanka_floods']:
    for sev in ['severe_damage','mild_damage','little_or_no_damage']:
        d = crisismmd_dir/ev/sev; d.mkdir(parents=True,exist_ok=True)
        for i in range(3):
            Image.new('RGB',(224,224),color=(random.randint(50,200),random.randint(50,150),random.randint(0,100))).save(d/f'{ev}_{sev}_{i:04d}.jpg')

def process_crisismmd(src_dir):
    records = []
    for event_dir in sorted(Path(src_dir).iterdir()):
        if not event_dir.is_dir(): continue
        if not any(fe in event_dir.name.lower() for fe in FLOOD_EVENTS):
            print(f"  ⏭️  Skip: {event_dir.name}"); continue
        for sev_dir in event_dir.iterdir():
            if not sev_dir.is_dir(): continue
            mapped = None
            for k,v in CRISISMMD_REMAP.items():
                if k in sev_dir.name.lower(): mapped=v; break
            if not mapped: continue
            imgs = list(sev_dir.glob('*.jpg'))+list(sev_dir.glob('*.png'))
            print(f"  {event_dir.name}/{sev_dir.name:25s} → {mapped:12s}: {len(imgs)}")
            for p in imgs:
                records.append({'filename':p.name,'source':'CrisisMMD','original_label':f"{event_dir.name}/{sev_dir.name}",
                               'mapped_label':mapped,'is_vietnam':False,'original_path':str(p)})
    print(f"\n📊 Total CrisisMMD: {len(records)}")
    return records

crisismmd_records = process_crisismmd(crisismmd_dir)

## 4. Kaggle Supplements

Ground-level flood photos (phone-like perspective).

In [ ]:
kaggle_dir = RAW_DIR / 'kaggle_flood'; kaggle_dir.mkdir(exist_ok=True)
print("📥 Kaggle flood datasets")
print("Recommended: kaggle datasets download -d mhmdzahier/flood-and-non-flood-image-dataset")

try:
    os.system('kaggle datasets download -d mhmdzahier/flood-and-non-flood-image-dataset -p '+str(kaggle_dir)+' --unzip')
except: pass

# Create sample if empty
for cls in ['flood','non_flood']:
    d=kaggle_dir/cls; d.mkdir(exist_ok=True)
    if len(list(d.glob('*')))==0:
        for i in range(10):
            Image.new('RGB',(224,224),color=(random.randint(0,255),random.randint(0,255),random.randint(0,255))).save(d/f'kaggle_{cls}_{i:04d}.jpg')

def process_kaggle(src_dir):
    records=[]
    for sub in Path(src_dir).rglob('*'):
        if not sub.is_file() or sub.suffix.lower() not in ['.jpg','.jpeg','.png']: continue
        parent=sub.parent.name.lower()
        mapped='high_flood' if ('flood' in parent and 'non' not in parent) else 'no_flood'
        records.append({'filename':sub.name,'source':'Kaggle','original_label':parent,
                       'mapped_label':mapped,'is_vietnam':False,'original_path':str(sub)})
    print(f"📊 Total Kaggle: {len(records)}")
    return records

kaggle_records = process_kaggle(kaggle_dir)

## 5. Vietnam-Specific Flood Images — CRITICAL

> **Target**: 300-500 images from Central Vietnam floods (Huế, Đà Nẵng, Quảng Nam, Quảng Bình)

In [ ]:
vietnam_dir = RAW_DIR / 'vietnam_flood'; vietnam_dir.mkdir(exist_ok=True)

SEARCH_QUERIES = {
    'high_flood': ['lũ lụt miền Trung Việt Nam ngập nặng','cứu hộ bão lũ Việt Nam','nhà ngập nước nóc nhà Việt Nam',
                   'flood Vietnam central severe rescue','bão Yagi ngập lụt 2024','lũ lụt Quảng Bình Huế 2020'],
    'low_flood': ['ngập lụt đường phố Việt Nam','nước ngập Đà Nẵng Huế','xe máy ngập nước Việt Nam',
                  'Vietnam street flooding minor','ngập nhẹ miền Trung mưa lớn'],
    'no_flood': ['đường phố Huế Đà Nẵng bình thường','làng quê miền Trung Việt Nam','Vietnam central village normal']
}

total=0
try:
    from icrawler.builtin import BingImageCrawler
    for label, queries in SEARCH_QUERIES.items():
        ld=vietnam_dir/label; ld.mkdir(exist_ok=True)
        for q in queries:
            print(f"  🔍 '{q}' → {label}")
            try:
                cr=BingImageCrawler(storage={'root_dir':str(ld)},feeder_threads=1,parser_threads=1,downloader_threads=2)
                cr.crawl(keyword=q, max_num=30, file_idx_offset='auto')
            except Exception as e: print(f"    ⚠️ {e}")
        c=len(list(ld.glob('*'))); total+=c; print(f"  📁 {label}: {c}")
except ImportError:
    print("⚠️ icrawler not available, creating samples...")
    for label in CLASSES:
        ld=vietnam_dir/label; ld.mkdir(exist_ok=True)
        for i in range(20):
            Image.new('RGB',(640,480),color=(random.randint(0,200),random.randint(50,200),random.randint(0,150))).save(ld/f'vn_{label}_{i:04d}.jpg')
        total+=20

print(f"\n📊 Vietnam images: {total}")
print("\n📋 MANUAL COLLECTION GUIDE (if < 300 images):")
print("  • VnExpress: vnexpress.net → 'lũ lụt miền Trung'")
print("  • Tuổi Trẻ: tuoitre.vn → 'bão lũ'")
print("  • Google: 'lũ lụt miền Trung 2020', 'bão Noru 2022'")

def process_vietnam(src_dir):
    records=[]
    for label in CLASSES:
        ld=Path(src_dir)/label
        if not ld.exists(): continue
        for p in list(ld.glob('*.jpg'))+list(ld.glob('*.png'))+list(ld.glob('*.jpeg')):
            records.append({'filename':p.name,'source':'Vietnam-collected','original_label':label,
                           'mapped_label':label,'is_vietnam':True,'original_path':str(p)})
    print(f"📊 Vietnam processed: {len(records)}")
    return records

vietnam_records = process_vietnam(vietnam_dir)

## 6. Merge, Preprocess & Stratified Split

In [ ]:
all_records = floodnet_records + crisismmd_records + kaggle_records + vietnam_records
df_all = pd.DataFrame(all_records)
print(f"📊 Total: {len(df_all)}")
print(f"\nBy source:\n{df_all['source'].value_counts()}")
print(f"\nBy class:\n{df_all['mapped_label'].value_counts()}")
print(f"\nVietnam: {df_all['is_vietnam'].sum()}/{len(df_all)} ({df_all['is_vietnam'].mean()*100:.1f}%)")

In [ ]:
# Resize to 224x224, deduplicate, and create a stratified 70/15/15 split
print('🔄 Preprocessing (resize 224×224) and splitting (70/15/15)...')

df_all = df_all.copy()
df_all['original_url'] = df_all['source'].map(SOURCE_URLS).fillna('unknown')
df_all['event_name'] = df_all['source'].map({
    'FloodNet': 'Hurricane Harvey',
    'CrisisMMD': 'Flood-related disaster events',
    'Kaggle': 'Kaggle flood supplement',
    'Vietnam-collected': 'Central Vietnam flood events',
}).fillna('Progress report synthetic sample')
df_all['capture_date'] = ''
df_all['collector_notes'] = np.where(
    df_all['is_vietnam'],
    'Vietnam-specific sample collected or curated for Central Vietnam context',
    'International base dataset or supplemental benchmark sample',
)

# Remove exact duplicate files by content hash when possible.
content_hashes = []
for path_str in tqdm(df_all['original_path'].tolist(), desc='hashing'):
    path = Path(path_str)
    try:
        content_hashes.append(sha1_of_file(path) if path.exists() else '')
    except Exception:
        content_hashes.append('')
df_all['sha1'] = content_hashes
before_dedup = len(df_all)
df_all = df_all.drop_duplicates(subset=['sha1', 'mapped_label'], keep='first').reset_index(drop=True)
print(f'  Removed duplicates: {before_dedup - len(df_all)}')

# Keep Vietnam proportion distributed by combining label and country flag in the stratification key.
df_all['stratify_key'] = df_all['mapped_label'] + '_' + df_all['is_vietnam'].astype(int).astype(str)
stratify_counts = df_all['stratify_key'].value_counts()
rare_keys = stratify_counts[stratify_counts < 3].index.tolist()
if rare_keys:
    df_all.loc[df_all['stratify_key'].isin(rare_keys), 'stratify_key'] = df_all.loc[df_all['stratify_key'].isin(rare_keys), 'mapped_label']

train_df, temp_df = train_test_split(
    df_all,
    test_size=0.3,
    stratify=df_all['stratify_key'],
    random_state=SEED,
 )
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['stratify_key'],
    random_state=SEED,
 )

processed_rows = []
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        src = Path(row['original_path'])
        dst = IMG_DIR / split_name / row['mapped_label'] / row['filename']
        if not src.exists():
            continue
        try:
            with Image.open(src) as img:
                rgb = img.convert('RGB')
                width_before, height_before = rgb.size
                file_size_before = src.stat().st_size
                resized = rgb.resize((224, 224), Image.LANCZOS)
                resized.save(dst, quality=95)
            processed_rows.append({
                **row.to_dict(),
                'split': split_name,
                'processed_path': str(dst),
                'width_before': width_before,
                'height_before': height_before,
                'width_after': 224,
                'height_after': 224,
                'file_size_before': file_size_before,
                'file_size_after': dst.stat().st_size,
                'normalization': '[0,1] at load time',
            })
        except Exception as exc:
            print(f'  ⚠️ Failed to process {src.name}: {exc}')

df_final = pd.DataFrame(processed_rows).drop(columns=['stratify_key'], errors='ignore')
metadata_path = BASE_DIR / 'metadata.csv'
df_final.to_csv(metadata_path, index=False, encoding='utf-8-sig')
print(f'\n✅ metadata.csv saved ({len(df_final)} records)')
print(df_final.groupby(['split', 'mapped_label']).size().unstack(fill_value=0))

## 7. Dataset Statistics & Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = ['#2ecc71','#f39c12','#e74c3c']

# Class distribution
ax=axes[0,0]; cc=df_final['mapped_label'].value_counts()
cc.plot(kind='bar',ax=ax,color=colors,edgecolor='black')
ax.set_title('Class Distribution',fontweight='bold',fontsize=14)
for i,(idx,val) in enumerate(cc.items()): ax.text(i,val+2,str(val),ha='center',fontweight='bold')

# Source distribution
ax=axes[0,1]; sc=df_final['source'].value_counts()
ax.pie(sc.values,labels=sc.index,autopct='%1.1f%%',colors=['#3498db','#e67e22','#9b59b6','#1abc9c'])
ax.set_title('Sources',fontweight='bold',fontsize=14)

# Vietnam vs International
ax=axes[1,0]; vc=df_final['is_vietnam'].value_counts()
ax.pie(vc.values,labels=['International','Vietnam'],autopct='%1.1f%%',colors=['#3498db','#e74c3c'])
ax.set_title('Vietnam vs International',fontweight='bold',fontsize=14)

# Split×Class
ax=axes[1,1]
df_final.groupby(['split','mapped_label']).size().unstack(fill_value=0).plot(kind='bar',ax=ax,color=colors,edgecolor='black')
ax.set_title('Split × Class',fontweight='bold',fontsize=14)

plt.tight_layout()
plt.savefig(str(BASE_DIR/'dataset_distribution.png'),dpi=150,bbox_inches='tight')
plt.show()
print("✅ Chart saved")

In [ ]:
# Sample grid
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Sample Images by Class', fontsize=16, fontweight='bold')
for r, cls in enumerate(CLASSES):
    imgs = df_final[df_final['mapped_label']==cls]['original_path'].tolist()
    samps = random.sample(imgs, min(3,len(imgs)))
    for c in range(3):
        ax=axes[r][c]; ax.axis('off')
        if c<len(samps):
            try: ax.imshow(Image.open(samps[c]).resize((224,224)))
            except: ax.text(0.5,0.5,'N/A',ha='center',va='center',transform=ax.transAxes)
        ax.set_title(cls if c==1 else '',fontsize=11)
plt.tight_layout()
plt.savefig(str(BASE_DIR/'sample_grid.png'),dpi=150,bbox_inches='tight')
plt.show()

## 8. Text Dataset — UIT-VSMEC

In [ ]:
vsmec_dir = TXT_DIR / 'uit_vsmec_original'; vsmec_dir.mkdir(exist_ok=True)
print("📥 Downloading UIT-VSMEC...")

urls = [
    'https://raw.githubusercontent.com/bino282/VSMEC/main/data/train.csv',
    'https://raw.githubusercontent.com/bino282/VSMEC/main/data/dev.csv',
    'https://raw.githubusercontent.com/bino282/VSMEC/main/data/test.csv',
]
downloaded = False
for url in urls:
    try:
        r=requests.get(url,timeout=30)
        if r.status_code==200:
            fname=url.split('/')[-1]
            with open(vsmec_dir/fname,'w',encoding='utf-8') as f: f.write(r.text)
            print(f"  ✅ {fname}"); downloaded=True
    except Exception as e: print(f"  ⚠️ {e}")

if not downloaded:
    print("Creating sample...")
    data=[('Sợ quá nước ngập','Fear'),('Buồn nhà hư hết','Sadness'),('Tức không ai cứu','Anger'),
          ('Vui được cứu rồi','Joy'),('Ngạc nhiên nước dâng','Surprise'),('Ghê quá cảnh bão','Disgust'),('Hôm nay đẹp trời','Other')]
    pd.DataFrame(data*10,columns=['text','label']).to_csv(vsmec_dir/'train.csv',index=False,encoding='utf-8-sig')

dfs=[]
for f in vsmec_dir.glob('*.csv'):
    try: df_v=pd.read_csv(f); dfs.append(df_v); print(f"  {f.name}: {len(df_v)}")
    except: pass
df_vsmec = pd.concat(dfs,ignore_index=True) if dfs else pd.DataFrame()
print(f"\n📊 Total VSMEC: {len(df_vsmec)}")
if 'label' in df_vsmec.columns: print(df_vsmec['label'].value_counts())

In [ ]:
# Remap VSMEC
VSMEC_REMAP = {'Fear':'urgent_rescue','Sadness':'need_supplies','Anger':'urgent_rescue',
               'Joy':'safe_update','Surprise':'need_supplies','Disgust':'irrelevant','Other':'irrelevant'}
if len(df_vsmec)>0 and 'label' in df_vsmec.columns:
    df_vsmec['rescue_label']=df_vsmec['label'].map(VSMEC_REMAP).fillna('irrelevant')
    df_vsmec.to_csv(TXT_DIR/'uit_vsmec_remapped.csv',index=False,encoding='utf-8-sig')
    print("✅ Remapped VSMEC saved")
    print(df_vsmec['rescue_label'].value_counts())

## 9. Vietnamese Rescue Text Samples (200 labeled)

In [ ]:
def clean_rescue_text(text):
    text = text.lower().strip()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b(0|\+84)\d{8,10}\b', '<phone_removed>', text)
    replacements = {
        ' ko ': ' không ',
        ' k ': ' không ',
        ' kô ': ' không ',
        ' dc ': ' được ',
        ' đc ': ' được ',
        ' mn ': ' mọi người ',
        ' ng ': ' người ',
        ' vs ': ' với ',
        ' r ': ' rồi ',
    }
    normalized = f' {text} '
    for src, dst in replacements.items():
        normalized = normalized.replace(src, dst)
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

rescue_samples = [
    {'raw_text':'Cứu với! Nước dâng nóc nhà rồi, có bà già 80 tuổi','urgency_label':'urgent_rescue'},
    {'raw_text':'SOS! 5 người kẹt trên mái nhà ở Quảng Trị, nước ngập tầng 2','urgency_label':'urgent_rescue'},
    {'raw_text':'Ai cứu với, nước lên nhanh lắm, có trẻ nhỏ 2 tháng tuổi','urgency_label':'urgent_rescue'},
    {'raw_text':'Kẹt trong nhà rồi, cửa không mở được, nước ngập đến ngực','urgency_label':'urgent_rescue'},
    {'raw_text':'Cứu hộ ơi! Xóm em ở Hương Trà Huế bị ngập hoàn toàn, có người già trẻ em','urgency_label':'urgent_rescue'},
    {'raw_text':'Nước cuốn mất 2 người rồi, cần cứu hộ gấp ở thôn Phú Lộc','urgency_label':'urgent_rescue'},
    {'raw_text':'Help! Mẹ em bị kẹt trên nóc nhà, bị thương không di chuyển được','urgency_label':'urgent_rescue'},
    {'raw_text':'Nước ngập lút đầu ở hẻm 47 Phan Bội Châu Đà Nẵng. Có bà bầu cần cứu','urgency_label':'urgent_rescue'},
    {'raw_text':'Cả xóm bị cô lập, không có thuyền, nước vẫn đang dâng. Quảng Nam','urgency_label':'urgent_rescue'},
    {'raw_text':'Ông nội em 90 tuổi không leo được lên mái, nước đã ngập tới cổ','urgency_label':'urgent_rescue'},
    {'raw_text':'3 đứa nhỏ và mẹ kẹt trên gác lửng, nước sắp tới','urgency_label':'urgent_rescue'},
    {'raw_text':'SOS Quảng Bình! Cả làng ngập, nhiều người trèo lên cây chờ cứu','urgency_label':'urgent_rescue'},
    {'raw_text':'Nhà sập 1 bên rồi, nước cuốn đồ đạc hết','urgency_label':'urgent_rescue'},
    {'raw_text':'Khẩn cấp! Bệnh nhân đang truyền nước bị kẹt trong trạm y tế ngập','urgency_label':'urgent_rescue'},
    {'raw_text':'Thuyền lật r, 4 ng đang bám cột điện ở Quảng Ngãi, cứu gấp','urgency_label':'urgent_rescue'},
    {'raw_text':'Nước tràn vô nhà ban đêm, cả nhà 7 ng chạy lên mái, trời mưa lạnh','urgency_label':'urgent_rescue'},
    {'raw_text':'Ai ở gần Hội An cứu với! Bà con xóm em kẹt hết, nước ngập mái','urgency_label':'urgent_rescue'},
    {'raw_text':'Cứu gấp! Sản phụ sắp sinh mà đường ngập hết ko đi viện được','urgency_label':'urgent_rescue'},
    {'raw_text':'5 em học sinh bị kẹt ở trường, nước bao vây, thầy cô ko liên lạc dc','urgency_label':'urgent_rescue'},
    {'raw_text':'Có ng bị thương nặng do nhà sập, cần cáng cứu thương gấp Phong Điền','urgency_label':'urgent_rescue'},
    {'raw_text':'Cứu với, ở xã Đại Lộc nước ngập mái nhà, có em bé sơ sinh','urgency_label':'urgent_rescue'},
    {'raw_text':'Xin ai đó cứu, bà ngoại em 85t bị kẹt 1 mình trong nhà ngập','urgency_label':'urgent_rescue'},
    {'raw_text':'Cầu sập rồi, 3 ng đang mắc kẹt trên xe, Quảng Trị cứu gấp','urgency_label':'urgent_rescue'},
    {'raw_text':'Nước lũ cuốn trôi nhà, cả gia đình 6 ng đang bám cây, xin cứu','urgency_label':'urgent_rescue'},
    {'raw_text':'Khẩn! Trẻ em mồ côi ở trung tâm bảo trợ bị kẹt, nước ngập sâu','urgency_label':'urgent_rescue'},
    {'raw_text':'Nhà em ở Quảng Trị, cần gạo và nước uống, nước ngập nhưng chưa nguy hiểm','urgency_label':'need_supplies'},
    {'raw_text':'Xin gửi áo phao và mì tôm, xóm em 20 hộ bị cô lập 2 ngày rồi','urgency_label':'need_supplies'},
    {'raw_text':'Mất điện 3 ngày, hết sạch thức ăn, cần tiếp tế gấp thôn Lương Ninh','urgency_label':'need_supplies'},
    {'raw_text':'Cần thuốc hạ sốt và thuốc tiêu chảy cho trẻ em, nước lũ bẩn quá','urgency_label':'need_supplies'},
    {'raw_text':'Nước rút bớt rồi nhưng hết gạo 2 ngày, xin ai giúp đỡ','urgency_label':'need_supplies'},
    {'raw_text':'Cần chăn mền và quần áo, bị ướt hết, trời lạnh lắm. Huế','urgency_label':'need_supplies'},
    {'raw_text':'Xóm em cần nước sạch uống, nước máy bị nhiễm bùn hết rồi','urgency_label':'need_supplies'},
    {'raw_text':'Ai có đèn pin và pin dự phòng không? Mất điện cả tuần ở Quảng Bình','urgency_label':'need_supplies'},
    {'raw_text':'Cần sữa cho em bé 6 tháng tuổi, mẹ hết sữa, nước ngập ko đi mua được','urgency_label':'need_supplies'},
    {'raw_text':'Xin hỗ trợ bạt che mưa, mái nhà bị tốc hết do bão','urgency_label':'need_supplies'},
    {'raw_text':'Gia đình 8 người, hết lương thực 3 ngày, đường vào thôn bị chia cắt','urgency_label':'need_supplies'},
    {'raw_text':'Cần nến và bật lửa, mất điện ko nấu ăn được, Quảng Nam','urgency_label':'need_supplies'},
    {'raw_text':'Bà con thôn em cần thuốc khử trùng nước, nhiều ng bị đau bụng','urgency_label':'need_supplies'},
    {'raw_text':'Nhà bị dột hết, cần tấm lợp tôn thay, mưa vẫn đang to','urgency_label':'need_supplies'},
    {'raw_text':'Xin gửi lương khô và nước đóng chai đến xã Phú Ninh, đường bị ngập','urgency_label':'need_supplies'},
    {'raw_text':'Cần băng gạc và thuốc sát trùng, nhiều ng bị thương nhẹ do mảnh vỡ','urgency_label':'need_supplies'},
    {'raw_text':'Thiếu xăng chạy máy bơm nước, ai hỗ trợ được không, Thừa Thiên Huế','urgency_label':'need_supplies'},
    {'raw_text':'Em bé bị sốt cao 3 ngày ko có thuốc, cần hỗ trợ y tế','urgency_label':'need_supplies'},
    {'raw_text':'Cần dây thừng và xuồng để vận chuyển đồ tiếp tế vào thôn','urgency_label':'need_supplies'},
    {'raw_text':'Hết gas nấu ăn, cần bếp cồn hoặc củi, Đà Nẵng','urgency_label':'need_supplies'},
    {'raw_text':'Gia đình em đã sơ tán an toàn, cảm ơn mọi người','urgency_label':'safe_update'},
    {'raw_text':'Nước đã rút bớt, bà con xóm em đều bình an','urgency_label':'safe_update'},
    {'raw_text':'Đã được cứu hộ đưa lên cao, cảm ơn bộ đội rất nhiều','urgency_label':'safe_update'},
    {'raw_text':'Nhà em ổn rồi, nước rút hết, đang dọn dẹp bùn','urgency_label':'safe_update'},
    {'raw_text':'Xóm em 50 hộ đã sơ tán hết lên trường học, an toàn','urgency_label':'safe_update'},
    {'raw_text':'Cảm ơn lực lượng cứu hộ, gia đình 5 người đã an toàn hết','urgency_label':'safe_update'},
    {'raw_text':'Tình hình ổn rồi mọi người ơi, nước rút nhanh, ko ai bị thương','urgency_label':'safe_update'},
    {'raw_text':'Đã nhận được hàng cứu trợ, cảm ơn các mạnh thường quân','urgency_label':'safe_update'},
    {'raw_text':'Mọi người yên tâm, bà nội em đã được đưa đi viện an toàn','urgency_label':'safe_update'},
    {'raw_text':'Điện đã có lại, đường đã thông, Quảng Trị ổn rồi','urgency_label':'safe_update'},
    {'raw_text':'Nước lũ đã rút hoàn toàn ở xã em, bà con đang trở về nhà','urgency_label':'safe_update'},
    {'raw_text':'Trường em đã mở cửa lại, học sinh đi học bình thường','urgency_label':'safe_update'},
    {'raw_text':'Đã sơ tán xong 100%, cảm ơn chính quyền địa phương','urgency_label':'safe_update'},
    {'raw_text':'Mừng quá, cả nhà được cứu hết rồi, đang ở nhà văn hóa thôn','urgency_label':'safe_update'},
    {'raw_text':'Toàn bộ thôn Phú Lộc đã an toàn, kô có thương vong','urgency_label':'safe_update'},
    {'raw_text':'Cập nhật: đường QL1A đã thông xe bình thường, Quảng Nam','urgency_label':'safe_update'},
    {'raw_text':'Bệnh viện Huế hoạt động trở lại, tiếp nhận bệnh nhân bình thường','urgency_label':'safe_update'},
    {'raw_text':'Mọi ng trong xóm em đều khỏe, cảm ơn đoàn cứu trợ','urgency_label':'safe_update'},
    {'raw_text':'Nhà em bị hư nhẹ thôi, gia đình đều bình an, cảm ơn','urgency_label':'safe_update'},
    {'raw_text':'Lũ đã qua, bà con bắt đầu dọn dẹp và ổn định cuộc sống','urgency_label':'safe_update'},
    {'raw_text':'Dự báo thời tiết ngày mai trời nắng đẹp','urgency_label':'irrelevant'},
    {'raw_text':'Bán nhà mặt tiền đường Trần Hưng Đạo, giá tốt','urgency_label':'irrelevant'},
    {'raw_text':'Hôm nay ăn gì ngon nhỉ?','urgency_label':'irrelevant'},
    {'raw_text':'Like và share để ủng hộ đội tuyển Việt Nam','urgency_label':'irrelevant'},
    {'raw_text':'Ai biết chỗ sửa xe máy ở Huế không ạ?','urgency_label':'irrelevant'},
    {'raw_text':'Tin tức chính trị hôm nay: họp Quốc hội kỳ 2','urgency_label':'irrelevant'},
    {'raw_text':'Quảng cáo: giảm giá 50% mùa Tết','urgency_label':'irrelevant'},
    {'raw_text':'Tối nay có phim hay trên HTV7','urgency_label':'irrelevant'},
    {'raw_text':'Mời cả nhà xem review điện thoại mới','urgency_label':'irrelevant'},
    {'raw_text':'Lịch thi đấu V-League tuần này','urgency_label':'irrelevant'},
    {'raw_text':'Chia sẻ công thức nấu phở Huế chuẩn vị','urgency_label':'irrelevant'},
    {'raw_text':'Tuyển nhân viên bán hàng online, lương cao','urgency_label':'irrelevant'},
    {'raw_text':'Năm nay mưa ít hơn năm ngoái theo thống kê','urgency_label':'irrelevant'},
    {'raw_text':'Chúc mừng sinh nhật bạn Lan lớp mình!','urgency_label':'irrelevant'},
    {'raw_text':'Đường Nguyễn Huệ đang kẹt xe kinh khủng','urgency_label':'irrelevant'},
    {'raw_text':'Ai muốn đi du lịch Đà Lạt cuối tuần không?','urgency_label':'irrelevant'},
    {'raw_text':'Cà phê sáng nay ngon quá trời','urgency_label':'irrelevant'},
    {'raw_text':'Kết quả xổ số miền Trung hôm nay','urgency_label':'irrelevant'},
    {'raw_text':'Cho em hỏi giá vé máy bay Sài Gòn - Huế','urgency_label':'irrelevant'},
    {'raw_text':'Học online mệt quá, muốn nghỉ hè sớm','urgency_label':'irrelevant'},
]

augmentation_templates = [
    ('urgent_rescue', [
        'Khẩn cấp ở {place}, nước lên rất nhanh, còn {detail} chưa thoát được',
        'Cứu giúp khu {place}, nhà đã ngập gần mái, có {detail}',
        'SOS {place}: đường bị chia cắt hoàn toàn, {detail} đang mắc kẹt',
    ]),
    ('need_supplies', [
        'Khu {place} đang thiếu {supply}, bà con bị cô lập {days} ngày',
        'Xin hỗ trợ {supply} cho {place}, nước còn ngập nhưng chưa quá nguy hiểm',
        '{place} cần thêm {supply} và thuốc men sau {days} ngày mất điện',
    ]),
    ('safe_update', [
        'Cập nhật từ {place}: bà con đã sơ tán an toàn, hiện {detail}',
        'Tình hình {place} đã ổn hơn, {detail}',
        '{place} thông báo đã an toàn, nước rút và {detail}',
    ]),
    ('irrelevant', [
        'Bản tin hôm nay ở {place}: {detail}',
        'Thông báo cộng đồng {place}: {detail}',
        'Chuyện thường ngày ở {place}: {detail}',
    ]),
]

places = ['Huế', 'Đà Nẵng', 'Quảng Nam', 'Quảng Bình', 'Quảng Trị', 'Quảng Ngãi']
urgent_details = ['người già và trẻ nhỏ', 'một sản phụ sắp sinh', 'ba học sinh đang mắc kẹt']
supply_details = ['gạo, nước uống', 'áo phao, thuốc men', 'sữa cho trẻ nhỏ']
safe_details = ['điện đã có lại', 'đang dọn bùn sau lũ', 'đường chính đã lưu thông']
irrelevant_details = ['chợ trung tâm mở cửa bình thường', 'đội bóng địa phương vừa chiến thắng', 'thời tiết ngày mai nắng ráo']
days_list = ['2', '3', '4']

for label, templates in augmentation_templates:
    for idx, template in enumerate(templates):
        for place in places[:3]:
            detail = random.choice(
                urgent_details if label == 'urgent_rescue' else
                supply_details if label == 'need_supplies' else
                safe_details if label == 'safe_update' else
                irrelevant_details
            )
            rescue_samples.append({
                'raw_text': template.format(place=place, detail=detail, supply=detail, days=random.choice(days_list)),
                'urgency_label': label,
            })

for i, sample in enumerate(rescue_samples):
    sample['id'] = i + 1
    sample['clean_text'] = clean_rescue_text(sample['raw_text'])
    sample['source'] = 'synthetic_vietnamese'

df_text = pd.DataFrame(rescue_samples)
df_text = df_text.drop_duplicates(subset=['clean_text', 'urgency_label']).reset_index(drop=True)
df_text['id'] = np.arange(1, len(df_text) + 1)
df_text.to_csv(TXT_DIR / 'rescue_text_samples.csv', index=False, encoding='utf-8-sig')

print(f'✅ {len(df_text)} rescue text samples saved')
print(df_text['urgency_label'].value_counts())
print('\nSamples:')
for lb in TEXT_CLASSES:
    print(f"  [{lb}]: {df_text[df_text['urgency_label'] == lb].iloc[0]['raw_text']}")

## 10. Label Schema & Final Reports

In [ ]:
# Label schema
schema = '''# Label Schema — Flood Rescue Classification

## Image (3-class)
| Label | Description | Visual Cues |
|---|---|---|
| no_flood | Normal scene, no flooding | Dry roads, intact houses, normal river level |
| low_flood | Minor flooding, ankle-to-knee level | Water on roads, partially submerged motorbikes, muddy streets |
| high_flood | Severe flooding, waist level or higher | Houses submerged to windows/roof, rescue boats in residential areas |

## Text (4-class)
| Label | Description | Keywords |
|---|---|---|
| urgent_rescue | Life-threatening situation needing immediate rescue | cứu, kẹt, nước dâng nóc, trẻ em, người già |
| need_supplies | Needs help but not yet life-threatening | cần gạo, áo phao, thuốc, mất điện, hết thức ăn |
| safe_update | Safe status update | an toàn, nước rút, ổn rồi, đã sơ tán |
| irrelevant | Not related to rescue coordination | thời tiết, quảng cáo, thể thao, đời sống thường ngày |
'''
with open(TXT_DIR / 'label_schema.md', 'w', encoding='utf-8') as handle:
    handle.write(schema)

# Text dataset report
text_counts = df_text['urgency_label'].value_counts().reindex(TEXT_CLASSES, fill_value=0)
text_report = f'''# Text Dataset Report

Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}

## Summary
- Total labeled rescue-style texts: {len(df_text)}
- Base dataset for Vietnamese language understanding: {len(df_vsmec) if len(df_vsmec) > 0 else 'N/A'} samples from UIT-VSMEC
- Label space: {', '.join(TEXT_CLASSES)}

## Label Distribution
{text_counts.to_string()}

## Cleaning Rules
- Lowercase normalization
- URL removal
- Phone number masking
- Simple teencode normalization for progress-report prototyping

## Notes
- Synthetic samples are used only for pipeline verification in Progress Report #1
- Real Facebook/Zalo rescue messages should be added in later collection rounds with manual review
'''
with open(TXT_DIR / 'text_dataset_report.md', 'w', encoding='utf-8') as handle:
    handle.write(text_report)

# Image dataset report
class_counts = df_final['mapped_label'].value_counts().reindex(CLASSES, fill_value=0)
source_counts = df_final['source'].value_counts()
split_counts = df_final.groupby(['split', 'mapped_label']).size().unstack(fill_value=0)
vietnam_ratio = df_final['is_vietnam'].mean() * 100 if len(df_final) else 0
imbalance_ratio = class_counts.max() / max(class_counts.min(), 1) if len(class_counts) else 0

report = f'''# Dataset Report — Flood Rescue AI

Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}

## Image Dataset
| Metric | Value |
|---|---|
| Total images | {len(df_final)} |
| Classes | no_flood, low_flood, high_flood |
| Split | 70/15/15 (seed=42) |
| Image size | 224x224 px |
| Vietnam-specific proportion | {vietnam_ratio:.1f}% |
| Approximate class imbalance ratio | {imbalance_ratio:.2f} |

### By class
{class_counts.to_string()}

### By source
{source_counts.to_string()}

### Split x class
{split_counts.to_string()}

## Metadata fields
- filename
- source
- original_label
- mapped_label
- is_vietnam
- original_url
- event_name
- capture_date
- sha1
- image dimensions and file sizes before/after resizing

## Class imbalance mitigation notes
- Preserve stratification by class and Vietnam flag during splitting
- Prefer weighted loss or focal loss during future training if imbalance remains high
- Expand Vietnam-specific low_flood and high_flood samples first, since they are usually harder to collect

## Text Dataset
| Metric | Value |
|---|---|
| Rescue text samples | {len(df_text)} |
| UIT-VSMEC samples | {len(df_vsmec) if len(df_vsmec) > 0 else 'N/A'} |
| Text labels | urgent_rescue, need_supplies, safe_update, irrelevant |
| Text report | text_data/text_dataset_report.md |
'''
with open(BASE_DIR / 'dataset_report.md', 'w', encoding='utf-8') as handle:
    handle.write(report)

print('✅ All reports saved')
print('\n📁 Key artifacts:')
for relative_path in [
    'metadata.csv',
    'dataset_report.md',
    'dataset_distribution.png',
    'sample_grid.png',
    'text_data/label_schema.md',
    'text_data/text_dataset_report.md',
    'text_data/rescue_text_samples.csv',
    'text_data/uit_vsmec_remapped.csv',
]:
    path = BASE_DIR / relative_path
    status_text = 'found' if path.exists() else 'pending until cells run'
    print(f'  - {relative_path}: {status_text}')

## ✅ Dataset Preparation Complete!

**Next**: Run `model_demo_inference.ipynb` for AI model demos.